In [ ]:
import pandas as pd
import numpy as np

import os
from sqlalchemy import create_engine
from datetime import datetime



In [ ]:
hostname = "AdhamNourMainPC"
dbname = "stg_personal_financial_analysis"
uname = "root"
pwd = "production_server"

In [ ]:
file_path=r"C:\Users\adham\OneDrive\Documents\Personal\Finance\Banking\CIB\Credit Cards\Platinum_202501.xls"

In [ ]:
df=pd.read_excel(io=file_path)

In [ ]:
index = df[df['Unnamed: 15'] == 'OPENING BALANCE'].index[0]+1
df=df.loc[index:]

In [ ]:
index = df[df['Unnamed: 15'] == 'CLOSING BALANCE'].index[0]-1
df=df.loc[:index]


In [ ]:
df = df.iloc[:, [4, 8, 15, 33]]
df

In [ ]:
df.columns = ["Transaction_Date","Value_Date","Description","Amount"]

In [ ]:
df

In [ ]:
for column in df.columns:
    if column != 'Amount':
        df[column] = df[column].fillna(method='ffill')


In [ ]:
df = df[df['Amount'].notna()]


In [ ]:
df['Transaction_Date'].replace('00-00',np.nan,inplace=True)
df['Value_Date'].replace('00-00',np.nan,inplace=True)

In [ ]:
df.to_excel(r"Platinum_202501_cleaned.xlsx", index=False)


In [ ]:
df['Amount']=df['Amount'].astype('str').str.strip()


In [ ]:
df[['Transaction_Amount', 'sign']] = df['Amount'].apply(
    lambda x: x.split(" ", 1) if " " in x else [x, None]
).apply(pd.Series)
df

In [ ]:
df['Transaction_Amount']=df['Transaction_Amount'].astype('float')
df['signed_amount'] = df['Transaction_Amount'].where(~df['sign'].isnull(), -1 * df['Transaction_Amount'])


In [ ]:
df = df.drop(columns=['Amount', 'Transaction_Amount', 'sign'])

In [ ]:
df['Transaction_Date'] = df['Transaction_Date'].fillna(method='ffill')
df['Value_Date'] = df['Value_Date'].fillna(method='ffill')

In [ ]:
df['currency_value'] = df['Description'].shift(-1)
df['signed_ammount_shift'] = df['signed_amount'].shift(-1)





In [ ]:
# Condition: signed_amont + signed_ammount_shift == signed_amont
mask = df['signed_amount'] + df['signed_ammount_shift'] != df['signed_amount']

# Update "Description" column by appending 'currency_value'
df.loc[mask, 'currency_value'] = np.nan


In [ ]:
df['currency']=df['currency_value'].astype('str').str[:3].replace('nan',np.nan);
df['value']=df['currency_value'].astype('str').str[3:].replace('nan',np.nan);


In [ ]:
df=df[df['signed_amount']!=0]


In [ ]:
df['value'] = pd.to_numeric(df['value'].str.replace(',', '.', regex=True), errors='coerce')

In [ ]:
filename = os.path.basename(file_path)

df['file_name']=filename


In [ ]:
df.drop(columns=['currency_value', 'signed_ammount_shift'], inplace=True)


In [ ]:
engine = create_engine(f"sqlite:///database.sqlite")

In [ ]:
df.to_sql("credit_card_transaction_files", engine, if_exists="replace", index=False)